# Transfer Learning with ProteinMPNN
In this example we will use the pretrained proteinMPNN to predict EC numbers for enzymes

# Install pacakges

In [ ]:
%pip install biopython
%pip install pandas
%pip install scikit-learn
%pip install matplotlib
%pip install torch
!git clone https://github.com/dauparas/ProteinMPNN.git

# Import packages

In [ ]:
import argparse
import copy
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

#path to ProteinMPNN repository
repo = Path("ProteinMPNN/")
checkpoint = str(repo / 'vanilla_model_weights' / 'v_48_020.pt')
#specify which ProteinMPNN model to use
model = 'vanilla_model_weight/v_48_020.pt'
#set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

#location of metadata and pdbs
metadata = "ec_prediction/metadata.csv"
pdb_dir = "ec_prediction/"

#epochs
epochs=20

output_dir = "./"
include_sequence = True
max_length = 1000
batch_size = 32
seed = 42
hidden = 132

# Set seed and load data

In [ ]:
AA = 'ACDEFGHIKLMNPQRSTVWY'


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def find_structure(row, pdb_dir):
    pid = str(row['protein_id']).strip()
    other = Path(str(row.get('structure_file', '')).replace('\\', '/')).name
    for p in [pdb_dir / f'{pid}.pdb', pdb_dir / other]:
        if p.is_file(): return p
    return None


def pdb_to_backbone(path, chain_id):
    """Single specified chain; skip residues lacking a complete N/CA/C/O backbone.
    Returns X [1,L,4,3], mask [1,L], residue_idx [1,L], chain_encoding [1,L].
    """
    model = next(PDBParser(QUIET=True).get_structure('p', str(path)).get_models())
    if chain_id not in model: raise ValueError(f'Chain {chain_id!r} not in {path.name}')
    coords, sequence = [], []
    for residue in model[chain_id]:
        aa = seq1(residue.resname, custom_map={'MSE': 'M'})
        if aa not in AA or not all(atom in residue for atom in ('N', 'CA', 'C', 'O')):
            continue
        xyz = np.stack([residue[a].coord for a in ('N', 'CA', 'C', 'O')]).astype('float32')
        if not np.isfinite(xyz).all(): continue
        coords.append(xyz)
        sequence.append(aa)
    if len(coords) < 2: raise ValueError('Fewer than 2 residues with complete backbone')
    n = len(coords)
    return (torch.from_numpy(np.stack(coords))[None],
            torch.ones(1, n, dtype=torch.float32),
            torch.arange(n, dtype=torch.long)[None],
            torch.ones(1, n, dtype=torch.long),
            ''.join(sequence))

#set seed
seed_all(42)

#load dataframe
df = pd.read_csv(metadata).dropna(subset=['protein_id', 'chain', 'label'])

#get classes
classes = sorted(df.label.unique().tolist())
label_map = {label: i for i, label in enumerate(classes)}

# Encode Structure with ProteinMPNN

In [ ]:
def load_encoder(repo, checkpoint, device):
    sys.path.insert(0, str(repo.resolve()))
    from protein_mpnn_utils import ProteinMPNN, gather_nodes
    ckpt = torch.load(str(checkpoint), map_location='cpu', weights_only=False)
    # Match the original ProteinMPNN inference script exactly.
    model = ProteinMPNN(num_letters=21, node_features=128, edge_features=128,
                        hidden_dim=128, num_encoder_layers=3, num_decoder_layers=3,
                        k_neighbors=ckpt['num_edges'], augment_eps=0.0, dropout=0.0)
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    model.to(device).eval().requires_grad_(False)
    return model, gather_nodes

@torch.inference_mode()
def encode_structure(model, gather_nodes, coords, mask, residx, chains, device):
    coords, mask, residx, chains = [v.to(device) for v in (coords, mask, residx, chains)]
    edges, edge_index = model.features(coords, mask, residx, chains)
    h_e = model.W_e(edges)
    h_v = torch.zeros(h_e.shape[0], h_e.shape[1], h_e.shape[-1], device=device)
    neighbor_mask = gather_nodes(mask.unsqueeze(-1), edge_index).squeeze(-1)
    attend = mask.unsqueeze(-1) * neighbor_mask
    for layer in model.encoder_layers:
        h_v, h_e = layer(h_v, h_e, edge_index, mask, attend)
    pooled = (h_v * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
    return pooled.squeeze(0).cpu().float()

def extract_embeddings(df, pdb_dir, model, gather_nodes, device):
    cache = Path(output_dir) / 'cached_proteinmpnn_embeddings.pt'
    # The cache is tied to the checkpoint, metadata IDs and data options;
    # delete/rebuild if the PDB contents have changed without changing their filenames.
    signature = {'checkpoint': str(Path(checkpoint).resolve()),
                 'checkpoint_mtime': Path(checkpoint).stat().st_mtime,
                 'pdb_dir': str(Path(pdb_dir).resolve()),
                 'records': [(str(r.protein_id), str(r.chain), str(r.label)) for r in df.itertuples(index=False)],
                 'use_sequence': include_sequence}
    if cache.is_file():
        saved = torch.load(cache, map_location='cpu', weights_only=False)
        if saved.get('signature') == signature:
            print(f'Loaded cached embeddings from {cache}')
            return saved['embeddings'], saved['records']
    embeddings, records, failures = [], [], Counter()
    pdb_dir = Path(pdb_dir)
    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        path = find_structure(row, pdb_dir)
        if path is None:
            failures['missing PDB'] += 1; continue
        try:
            X, mask, res_idx, chain_labels, seq = pdb_to_backbone(path, str(row['chain']).strip())
            if X.shape[1] > max_length:
                failures['too long'] += 1; continue
            vector = encode_structure(model, gather_nodes, X, mask, res_idx, chain_labels, device)
            if include_sequence:
                freq = torch.tensor([seq.count(aa) / len(seq) for aa in AA], dtype=torch.float32)
                vector = torch.cat([vector, freq])
            embeddings.append(vector)
            records.append({'protein_id': row['protein_id'], 'label': row['label'],
                            'chain': row['chain'], 'n_residues': len(seq), 'structure_file': str(path)})
        except (ValueError, KeyError, RuntimeError) as exc:
            failures[str(exc)[:100]] += 1
        if idx % 100 == 0: print(f'Processed {idx}/{len(df)} PDBs; usable {len(records)}')
    if not embeddings: raise RuntimeError(f'No structures successfully encoded. Errors: {failures}')
    if failures: print('Skipped structures:', dict(failures))
    tensor = torch.stack(embeddings)
    torch.save({'signature': signature, 'embeddings': tensor, 'records': records}, cache)
    print('Cached embeddings:', tuple(tensor.shape))
    return tensor, records

print(pdb_dir)
#load the ProteinMPNN encoder model
encoder, gather_nodes = load_encoder(repo, checkpoint, device)

#extract ProteinMPNN embeddings
#an embedding is a representation of data in another vector space
#it can refer to single layer determined embeddings we've discussed before or embeddings output by more complex neural networks
embeddings, records = extract_embeddings(df, pdb_dir, encoder, gather_nodes, device)

#delete encoder to free memory
del encoder

if torch.cuda.is_available(): torch.cuda.empty_cache()
y = np.array([label_map[r['label']] for r in records], dtype=np.int64)
print('EC classes:', classes, 'Counts:', dict(Counter(r['label'] for r in records)))
indices = np.arange(len(y))

#split data
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=seed, stratify=y)

# Normalization based on TRAINING embeddings only.
mean = embeddings[train_idx].mean(0)
std = embeddings[train_idx].std(0, unbiased=False).clamp(min=1e-5)
X = (embeddings - mean) / std
X = X.clamp(-10, 10)

#dataloader
def make_loader(ids, shuffle):
    return DataLoader(TensorDataset(X[ids], torch.tensor(y[ids])), batch_size=batch_size, shuffle=shuffle)
tr, va = make_loader(train_idx, True), make_loader(val_idx, False)

# Head to predict EC Number and Training

In [ ]:
class ECHead(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, n_classes))
    def forward(self, x): return self.net(x)


def epoch_pass(head, loader, device, optimizer=None):
    train = optimizer is not None
    head.train(train)
    losses, true, pred = [], [], []
    criterion = nn.CrossEntropyLoss()
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train: optimizer.zero_grad()
            logits = head(x)
            loss = criterion(logits, y)
            if train: loss.backward(); optimizer.step()
            losses.append((loss.item(), len(y)))
            true.extend(y.cpu().tolist()); pred.extend(logits.argmax(-1).detach().cpu().tolist())
    return {'loss': sum(a*n for a,n in losses)/sum(n for _,n in losses),
            'accuracy': accuracy_score(true, pred),
            'macro_f1': f1_score(true, pred, average='macro', zero_division=0),
            'y_true': true, 'y_pred': pred}

head = ECHead(X.shape[-1], len(classes), hidden).to(device)
optim = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-3)
history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}

for epoch in range(1, epochs+1):
        a = epoch_pass(head, tr, device, optim)
        b = epoch_pass(head, va, device)
        for name, data in [('train', a), ('val', b)]:
            history[f'{name}_loss'].append(data['loss'])
            history[f'{name}_accuracy'].append(data['accuracy'])
        if epoch == 1 or epoch % 10 == 0:
            print(f'Epoch {epoch:3d}: train loss {a["loss"]:.3f}, val loss {b["loss"]:.3f}, val acc {b["accuracy"]:.3f}')
        
result = epoch_pass(head, va, device)
print(classification_report(result['y_true'], result['y_pred'], labels=list(range(len(classes))), target_names=classes, zero_division=0))


# Plot training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric in zip(axes, ['loss','accuracy']):
    for split in ['train','val']:
        ax.plot(range(1,len(history[f'{split}_{metric}'])+1),history[f'{split}_{metric}'], label=split)
    ax.set(xlabel='Epoch', ylabel=metric.capitalize(), title=f'EC classifier {metric}')
    ax.legend(); ax.grid(alpha=.25)
fig.tight_layout()